# Tutorial 2: the engine in your own interface

This tutorial is for someone building an interface around the viewer: an operator window, a lab dashboard, or a script that wants a picture of a run while it is being written.

You use the **engine**, the Python package `zmart_viewer`, and bring your own page. The viewer's own window is not involved. The ZMART operator window in [ZMART Microscopy](https://github.com/thomdehoog/ZMART-microscopy) is built this way.

You will:

1. start the engine,
2. open a run as a *published* acquisition,
3. announce positions as a pretend microscope writes them,
4. put the picture on a page,
5. read back what was published, and write a projection.

You need the viewer installed. No microscope: a few lines of Python play one.

## What the engine is

A small HTTP server on the local machine. It reads OME-Zarr images, places each position where it belongs, follows the folder while a microscope writes, and serves the pieces of the picture a page asks for. **The engine reads and serves. Your page draws.** Because they talk over HTTP, the page can be written in any language.

The names below stay stable between versions. Everything deeper inside the package may move.

| What | Where | Used for |
|---|---|---|
| `make_server(...)` | `zmart_viewer` | Starting the engine on a port |
| `POST /api/stores/open` | HTTP | Telling the engine what to show |
| `POST /api/announce` | HTTP | Telling it a position was written |
| `GET /embedding.js` | HTTP | Putting the named views on your own canvas |
| `PublishedAcquisition`, `STORE` | `zmart_viewer` | Reading what the engine has published |
| `write_projection(...)` | `zmart_viewer` | Writing a Top, Min, Max or Sum view beside an image |

## 1. Set up

`sample_data.py` sits next to this notebook and writes small synthetic OME-Zarr images. `write_position()` below plays the microscope: it writes one position of a survey, each where the stage took it. `post()` sends one JSON request to the engine.

In [ ]:
import json
import threading
import urllib.error
import urllib.request
from pathlib import Path

from sample_data import a_specimen, write_image
from zmart_viewer import STORE, PublishedAcquisition, make_server, write_projection

run = Path("tutorial-data/interface-run").resolve()
positions = run / "positions"

STEP_UM = 224.0
PLACES = [(0, 0), (0, 1), (1, 0), (1, 1)]  # (row, column) on the stage


def write_position(number, *, revision=1):
    """The microscope writes one position. Returns the file name."""
    row, column = PLACES[number]
    store = write_image(
        positions / f"pos{number:02d}.ome.zarr",
        a_specimen(seed=number + 10 * revision),
        origin_um=(row * STEP_UM, column * STEP_UM),
    )
    return store.name


def post(route, payload):
    """One JSON request to the engine. Returns (status, answer)."""
    request = urllib.request.Request(
        f"{address}{route}",
        data=json.dumps(payload).encode(),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    try:
        with urllib.request.urlopen(request, timeout=600) as answer:
            return answer.status, json.loads(answer.read() or b"{}")
    except urllib.error.HTTPError as refusal:
        return refusal.code, json.loads(refusal.read() or b"{}")

## 2. Start the engine

`make_server()` builds the server but does not run it. You choose the thread.

| Argument | Means |
|---|---|
| `port=0` | Let the machine pick a free port |
| `data_dir` | The folder the load window starts in, and where marks are saved |
| `live=True` | Keep following the folder while images arrive |
| `allow_open=True` | Let a page open folders through the engine |
| `transparent_background=True` | Draw nothing where nothing was acquired yet |

In [ ]:
server = make_server(
    port=0,
    data_dir=run,
    live=True,
    allow_open=True,
    transparent_background=True,
)
threading.Thread(target=server.serve_forever, daemon=True).start()
address = f"http://127.0.0.1:{server.server_address[1]}"

with urllib.request.urlopen(f"{address}/api/health") as answer:
    print(address, json.loads(answer.read()))

The engine is up and serves an empty picture. It waits to be told what to show.

## 3. Tell it what to show

The microscope writes its first position. Then open the folder of positions as a **published acquisition**: one the engine keeps up to date as you tell it about new writes.

| Field | Means |
|---|---|
| `path` | The folder of positions |
| `bake` | `true` makes it a published acquisition. The engine keeps a coarse overview of the whole canvas as files beside the run, in `.zmart-viewer/`, so a survey of thousands of positions opens fast |
| `canvas` | The stage area the run will cover, in micrometres. The picture is laid out on it from the start, so the view never jumps as positions arrive |
| `source_revisions` | Which positions are complete, and a revision number for each |

Your own image files are never changed. The overview is made by averaging, so each position's smaller copies must be averages too: its OME-Zarr description says `"type": "mean"`. `sample_data.write_image()` writes that.

In [ ]:
revisions = {write_position(0): 1}
canvas = {"x_um": [0, STEP_UM + 256], "y_um": [0, STEP_UM + 256]}

status, answer = post(
    "/api/stores/open",
    {
        "path": str(positions),
        "bake": True,
        "canvas": canvas,
        "source_revisions": revisions,
    },
)
print(status, answer.get("error") or f"showing {answer['groups']}")
print("Open this address in your browser to watch:", address)

## 4. Announce as the microscope writes

After every completed write, send the **full** current mapping of position to revision. A revision is any number that goes up when a position's pixels change.

The answer's `told` is how many open pages were told to refresh.

In [ ]:
for number in (1, 2, 3):
    revisions[write_position(number)] = 1
    status, answer = post(
        "/api/announce",
        {"publications": [{"path": str(positions), "source_revisions": revisions}]},
    )
    print(f"position {number} announced:", status, answer)

A position was rewritten? Raise its revision and announce again. An unchanged mapping does no work.

In [ ]:
revisions[write_position(0, revision=2)] = 2
print("rewritten:", post("/api/announce", {"publications": [{"path": str(positions), "source_revisions": revisions}]}))
print("unchanged:", post("/api/announce", {"publications": [{"path": str(positions), "source_revisions": revisions}]}))

You do not have to say *what* changed inside a position. The engine reads that from the files.

Announcing is not compulsory. The engine also watches the folder, which is what makes it work with a microscope that has never heard of ZMART. But announcing is better: the watcher can only guess that a write has finished, and your software knows.

**Send announcements from a background thread** of the software that wrote the data, not from inside a capture or status callback. A slow page must never hold up the microscope.

## 5. Put the picture on your page

The engine serves `/embedding.js`, a plain ES module with no dependencies. It tells your page which named views (Slice, Top, Min, Max, Sum) exist for each acquisition and picks one, from the declared metadata and not from file names.

In [ ]:
with urllib.request.urlopen(f"{address}/embedding.js") as answer:
    script = answer.read().decode()
print("\n".join(line for line in script.splitlines() if line.startswith("export function")))

Your page imports it like this, and checks the version first:

```html
<script type="module">
  import { EMBEDDING_API_VERSION, viewChoices, selectedViews, keepDepthLocal }
    from "http://127.0.0.1:PORT/embedding.js";
  if (EMBEDDING_API_VERSION !== 1) throw new Error("newer viewer than this page knows");
</script>
```

The picture itself is drawn by [neuroglancer](https://github.com/google/neuroglancer), the one drawing engine the viewer supports. Your page creates its own neuroglancer, with the growth patches the engine ships in `neuroglancer-growth.mjs`, and feeds it the views the embedding script selects. The full description, including how Z-slider transitions are held so the old plane is never shown as the new one, is in [the embedding API](../embedding.md).

Your interface brings its own controls: sliders, channel settings, and whatever else your experiment needs.

## 6. Read what was published, from Python

Your own code may want to know what the engine made of an acquisition, for instance to check that a run is complete or to save a thumbnail. `PublishedAcquisition` reads the engine's output beside the run. `STORE` is the overview's path inside it.

In [ ]:
print("the overview lives at", positions / STORE)

view = PublishedAcquisition(positions, piece=64)
print("published pictures:", view.sources)
print("revision:", view.revision, "(goes up whenever any of them changes)")
view.close()

## 7. Write a projection

`write_projection()` writes a Min, Max or Sum projection of one position beside it, for example for an analysis step to look at. The output records which source, method and revision made it, so unchanged inputs are no-ops and nothing else ever overwrites it.

In [ ]:
source = positions / "pos00.ome.zarr"
target = run / "projections" / "pos00-max.ome.zarr"

written = write_projection(source, target, "max", revision=2)
print("written:", written.relative_to(run))

again = write_projection(source, target, "max", revision=2)
print("same inputs again, nothing rewritten:", again == written)

## 8. Stop the engine

In [ ]:
server.shutdown()
server.server_close()

## What stays with you

The engine is deliberately general. It knows nothing about stages, targets, drivers or workflow steps. Those belong to your interface:

- drawing your own marks over the picture: targets, the stage position, the field of view;
- turning a click on the picture into a stage coordinate;
- deciding which folder is the current run, and when a run starts or ends.

That line is what lets one engine serve every microscope behind one operator window, and still be a plain viewer for anyone else.

Read next: [the embedding API](../embedding.md), [the named views](../view_modes.md), and [how the viewer is built](../how_it_works/ARCHITECTURE.md).